<a href="https://colab.research.google.com/github/SNK005/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SNK005/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [21]:
%pip -q install duckdb huggingface_hub

import duckdb
from google.colab import userdata

token = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{token}')"
)

print("DuckDB connected to Hugging Face.")

DuckDB connected to Hugging Face.


One row represents one content item for one client on one report date in the warehouse. For this Lane 2 analysis, we use March 2026 as the development month and aggregate the available performance data for each content item to create features for the content-review ranking task.

**Query #1 — Grain**

In [22]:
REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {REL}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,n


**Verification result:** No duplicate report_date + client_hash_id + content_hash_id combinations were found in March 2026. This supports the stated daily grain of one row per content item, client, and report date.

**Query #2 — Row count + date span**

In [23]:
con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {REL}
""").df()

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


**Verification result:** The March 2026 slice contains 9,841,378 rows and covers dates from 2026-03-01 through 2026-03-31.

**Query #3 — Availability**

In [24]:
con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS available_rows
    FROM {REL}
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,available_rows
0,9841378,3611061


**Verification result:** My first attempt filtered on client_has_gsc, which turned out to be a client-level flag (true for any row belonging to a GSC-enabled client), not a row-level indicator — that's why it returned 100% availability, which didn't match the lane guide's own density figures. The correct row-level column is gsc_data_available. Filtering on that shows 3,611,061 of 9,841,378 rows (36.7%) actually have GSC data available for that specific day — consistent with the lane guide's overall figure that roughly 37% of daily rows across the full warehouse have GSC impressions

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature:
- impressions_month (sum of gsc_impressions) — search visibility available before review prioritization.
- clicks_month (sum of gsc_clicks) — search traffic available before review prioritization.
- avg_position_month (avg of gsc_avg_position) — search ranking signal available before review prioritization.
- engagement_rate_month (ga4_engaged_sessions / ga4_sessions) — engagement signal available before review prioritization.
- ai_sessions_month (sum of sessions_ai) — AI-referral session volume available before review prioritization; expected to be sparse.

Label / proxy:
- No label currently exists in this warehouse table. Unlike the starter CSV, this table has no trend_direction or trend_pct column. A declining/opportunity label would need to be constructed from the daily fact history, such as comparing an earlier performance window with a later window. That is future work and is not available in this month's slice as-is.

Context:
- report_date — identifies the observation date.
- client_hash_id — used for client grouping/splitting, not as a model feature.
- content_hash_id — identifies the content item, not as a model feature.
- month — identifies the warehouse partition.
- gsc_data_available — used to determine whether GSC metrics are available.
- ga4_data_available — used to determine whether GA4 metrics are available.

Excluded:
- client_has_gsc and client_has_ga4 — client-level integration flags, not row-level performance availability.
- Any field derived from a label or future outcome — excluded to prevent data leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

1. **`impressions_month`** — knowable at the decision moment because it's a sum of search impressions that already occurred in the past window; nothing about it depends on future data.
2. **`clicks_month`** — knowable at the decision moment because it's a sum of already-recorded past search clicks.
3. **`avg_position_month`** — knowable at the decision moment because it's an average of already-observed daily search rankings from the past window.
4. **`engagement_rate_month`** — knowable at the decision moment because it's computed only from `ga4_engaged_sessions` and `ga4_sessions`, both already-recorded past analytics events; however, it is unusable in this March 2026 feature frame because it is missing for all 241,200 content rows.
5. **`ai_sessions_month`** — knowable at the decision moment because it's a sum of already-recorded past AI-referral sessions; however, coverage is extremely sparse for this month, with 240,948 of 241,200 rows (99.9%) missing, which limits its practical usefulness right now.

In [25]:
feature_sql = f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_impressions END) AS impressions_month,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_clicks END) AS clicks_month,
    AVG(CASE WHEN gsc_data_available IS TRUE THEN gsc_avg_position END) AS avg_position_month,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions END)
        / NULLIF(SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions END), 0)
        AS engagement_rate_month,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN sessions_ai END) AS ai_sessions_month
FROM {REL}
GROUP BY client_hash_id, content_hash_id
"""

features = con.sql(feature_sql).df()
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,impressions_month,clicks_month,avg_position_month,engagement_rate_month,ai_sessions_month
0,client_62f4a7e64f5e0096,content_d0dff76c889de68f,181.0,0.0,5.147402,NaN,NaN
1,client_62f4a7e64f5e0096,content_67741cce996cfafa,46.0,1.0,4.828125,NaN,NaN
2,client_62f4a7e64f5e0096,content_2e6360ad20fd7107,899.0,1.0,5.145765,NaN,NaN
3,client_62f4a7e64f5e0096,content_ac8663da7484669a,34.0,0.0,4.909314,NaN,NaN
4,client_62f4a7e64f5e0096,content_65c50dfe9d87a585,3108.0,0.0,6.969536,NaN,NaN


In [26]:
features.isna().sum()

,0
client_hash_id,0
content_hash_id,0
impressions_month,154699
clicks_month,154699
avg_position_month,154699
engagement_rate_month,241200
ai_sessions_month,240948


In [27]:
features = features[features["clicks_month"].notna()].copy()

features["demo_label"] = (
    features["clicks_month"] > features["clicks_month"].median()
).astype(int)

features["demo_label"].value_counts()

,count
demo_label,
0,107901
1,68837


In [28]:
#Leaky model
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X = features[["impressions_month", "clicks_month", "avg_position_month",
              "engagement_rate_month", "ai_sessions_month", "demo_label"]].fillna(0)
y = features["demo_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Leaky test accuracy:", round(accuracy_score(y_test, pred), 3))

Leaky test accuracy: 1.0


### Leakage remains through the label-derived source feature

Removing the literal `demo_label` column is not sufficient because `demo_label` was constructed directly from `clicks_month`. Keeping `clicks_month` therefore still leaks information about the target.

In [29]:
#Still leaky
X = features[["impressions_month", "clicks_month", "avg_position_month",
              "engagement_rate_month", "ai_sessions_month"]].fillna(0)
y = features["demo_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Honest test accuracy:", round(accuracy_score(y_test, pred), 3))

Honest test accuracy: 1.0


### Leakage-free demonstration

The label and the feature used to construct it (`clicks_month`) are both excluded from the model inputs. The remaining features are used only as a synthetic demonstration of a leakage-free evaluation.

In [30]:
#Actually leakage-free
X = features[["impressions_month", "avg_position_month",
              "engagement_rate_month", "ai_sessions_month"]].fillna(0)
y = features["demo_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = DecisionTreeClassifier(max_depth=3, random_state=42)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Leakage-free test accuracy:", round(accuracy_score(y_test, pred), 3))

Leakage-free test accuracy: 0.841


**Leakage lesson**

The deliberately leaked model achieved a test accuracy of 1.000 because the target was directly included as an input feature. Removing only the target column was still insufficient because `demo_label` had been constructed directly from `clicks_month`, so `clicks_month` also leaked the target and produced another 1.000 score. After removing both the label and the feature used to construct it, the leakage-free demonstration achieved a test accuracy of 0.845.

This is a synthetic leakage demonstration, not a validated performance result for the Lane 2 decline/opportunity task. The actual warehouse table does not currently contain a decline label, so a real future-outcome label would need to be constructed from historical windows before evaluating a real ranking model.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Limitation — no existing decline/outcome label:** The March 2026 warehouse slice contains daily performance facts and availability flags, but no `trend_direction`, `trend_pct`, or other ready-made decline label. Therefore, the current five-feature frame can describe content performance but cannot yet be evaluated as a true decline/opportunity prediction model. A future modeling version would need to construct a target from separate historical and future windows, while keeping future outcome information out of the features.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.